# 原模型诊断
先测重复调用噪声，再测无关上下文与大 K；不预设原模型不稳定。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [ ]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


先核对 OpenJev 完整产物：固定 revision 的 manifest、各文件大小和 SHA-256。只有缺失/损坏文件才重下，坏缓存入口保留为 .invalid-*；不删除 00 数据或检索缓存。Colab 从 Drive 缓存复制到本机临时目录，以普通文件加载；约需额外 9 GB 本地磁盘，复制有进度，重新连接需要重新复制，通常不需要重新下载。此格不分配 4B GPU 模型。

In [ ]:
import importlib
from sa_jev import artifact_cache, model, experiments
for module in (artifact_cache, model, experiments):
    importlib.reload(module)
verified_artifact = artifact_cache.prepare_openjev_artifact(config['model']['artifact'], config['model'].get('revision'))
print('Verified OpenJev artifact:', verified_artifact)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

Keep the prepared full data root. Start with 2 questions; set RUN_FULL_DIAGNOSTICS=True only after status=ok and measured latency/memory are available. Run repeat diagnostics and K efficiency in separate cells.

In [ ]:
# Keep MODE/ROOT='full' to reuse the completed data; control run size here.
import copy, importlib, torch
from sa_jev import io, model, evaluation, experiments
for module in (io, model, evaluation, experiments):
    importlib.reload(module)
RUN_FULL_DIAGNOSTICS = True  # Set True after the 2-question check succeeds.
DIAGNOSTIC_LIMIT = (8 if MODE == 'pilot' else 100) if RUN_FULL_DIAGNOSTICS else 2
efficiency_config = copy.deepcopy(config)
if not RUN_FULL_DIAGNOSTICS:
    efficiency_config['evaluation']['k_values'] = [16]
print('ROOT:', ROOT, 'questions:', DIAGNOSTIC_LIMIT,
      'repeats:', config['evaluation']['repeats'],
      'efficiency K:', efficiency_config['evaluation']['k_values'], flush=True)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          'native BF16:', torch.cuda.is_bf16_supported(including_emulation=False), flush=True)
    if not torch.cuda.is_bf16_supported(including_emulation=False):
        print('This BF16 recipe is not validated on T4. Prefer A100 for the 4B experiments.', flush=True)
else:
    print('No CUDA GPU detected. Select a GPU runtime for OpenJev-4B.', flush=True)


ROOT: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full questions: 100 repeats: 3 efficiency K: [16, 32, 64, 128, 256]
GPU: NVIDIA A100-SXM4-40GB native BF16: True


In [ ]:
from sa_jev.experiments import diagnostics_experiment, efficiency_experiment
diagnostic_rows = diagnostics_experiment(ROOT, config, limit=DIAGNOSTIC_LIMIT)

[diagnostics] Reading questions and cached pools; method=openjev_frozen, limit=100
[diagnostics] Prepared 100 states; loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 12.5s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[diagnostics] Model ready on cuda:0
[diagnostics] 100 questions x 3 repeats = 300 forwards; device=cuda:0
[diagnostics] START 1/300: question=1/100 repeat=1/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 1/300: status=ok seconds=5.1 ETA=1532.7s peak_reserved_gib=8.083984375
[diagnostics] START 2/300: question=1/100 repeat=2/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 2/300: status=ok seconds=2.9 ETA=1196.7s peak_reserved_gib=8.083984375
[diagnostics] START 3/300: question=1/100 repeat=3/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 3/300: status=ok seconds=2.8 ETA=1076.6s peak_reserved_gib=8.083984375
[diagnostics] START 4/300: question=2/100 repeat=1/3 K=32 id=hotpotqa:5ae1f17e554299234fd0434

In [7]:
efficiency_rows = efficiency_experiment(ROOT, efficiency_config, limit=DIAGNOSTIC_LIMIT)

[efficiency] Preparing openjev_frozen; requested question limit=100
[efficiency] 100 questions per K; K=[16, 32, 64, 128, 256]; loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 4.6s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[efficiency] Model ready on cuda:0
[efficiency] START K=16: 100 questions; each may take up to 4 READ decisions
openjev_frozen 1/100 status=ok ETA=170.0s peak_reserved_gib=8.0859375
openjev_frozen 2/100 status=ok ETA=167.8s peak_reserved_gib=8.0859375
openjev_frozen 3/100 status=ok ETA=166.5s peak_reserved_gib=8.11328125
openjev_frozen 4/100 status=ok ETA=165.1s peak_reserved_gib=8.11328125
openjev_frozen 5/100 status=ok ETA=163.2s peak_reserved_gib=8.11328125
openjev_frozen 6/100 status=ok ETA=224.6s peak_reserved_gib=8.15234375
openjev_frozen 7/100 status=ok ETA=268.1s peak_reserved_gib=8.15234375
openjev_frozen 8/100 status=ok ETA=251.7s peak_reserved_gib=8.15234375
openjev_frozen 9/100 status=ok ETA=266.8s peak_reserved_gib=8.15234375
openje

In [ ]:
if os.environ.get('TYPESAFE_API_KEY') or os.environ.get('JEV_API_KEY'):
    diagnostics_experiment(ROOT, config, method='dynamic_jev', limit=DIAGNOSTIC_LIMIT)
    efficiency_experiment(ROOT, efficiency_config, method='dynamic_jev', limit=DIAGNOSTIC_LIMIT)
else:
    print('Official Jev API unavailable; no official-Jev result is reported.')

[diagnostics] Reading questions and cached pools; method=dynamic_jev, limit=100
[diagnostics] Prepared 100 states; loading model...
[diagnostics] Model ready on cpu
[diagnostics] 100 questions x 3 repeats = 300 forwards; device=cpu
[diagnostics] START 1/300: question=1/100 repeat=1/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 1/300: status=ok seconds=0.3 ETA=108.5s peak_reserved_gib=None
[diagnostics] START 2/300: question=1/100 repeat=2/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 2/300: status=ok seconds=0.3 ETA=103.7s peak_reserved_gib=None
[diagnostics] START 3/300: question=1/100 repeat=3/3 K=32 id=hotpotqa:5a85c3225542992a431d1b95
[diagnostics] DONE 3/300: status=ok seconds=0.3 ETA=102.6s peak_reserved_gib=None
[diagnostics] START 4/300: question=2/100 repeat=1/3 K=32 id=hotpotqa:5ae1f17e554299234fd04349
[diagnostics] DONE 4/300: status=ok seconds=0.3 ETA=99.6s peak_reserved_gib=None
[diagnostics] START 5/300: question=2/100 repeat=2/3 K=32 id=

无关状态干扰使用独立且经过核验的测试包，保持不调参。未核验草稿不会被自动当成测试结果。

In [ ]:
from sa_jev.experiments import robustness_experiment
if (ROOT/'data/perturbations_reviewed.jsonl').exists():
    robustness_experiment(ROOT, config, tag='original_perturbations', methods=['openjev_frozen'] + (['dynamic_jev'] if os.environ.get('TYPESAFE_API_KEY') else []))
else:
    print('Awaiting reviewed independent perturbation packets; repeat/K diagnostics above are separate measurements.')

Awaiting reviewed independent perturbation packets; repeat/K diagnostics above are separate measurements.


In [ ]:
# Historical two-question check: disabled after formal evaluation to preserve results.
print('Skipped the old 2-question efficiency rerun. Run the read-only result audit below instead.')

[efficiency] Preparing openjev_frozen; requested question limit=2
[efficiency] 2 questions per K; K=[16, 32, 64, 128, 256]; loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[OpenJev] Checking artifact: /content/drive/MyDrive/SA-Jev/huggingface/hub/models--shenjunhao--OpenJev-4B/snapshots/064bd435c6713486450194e29d526266b28941e2
[OpenJev] Loading verified ordinary files from local runtime: /tmp/sajev_openjev_models/fd2ba2d05d9eb7ca5bbc3411
[OpenJev] 1/11 backbone/config.json: expected 1,977 bytes; checking SHA-256...
[OpenJev] 2/11 backbone/model-00001-of-00003.safetensors: expected 3,991,295,368 bytes; checking SHA-256...
[OpenJev] 3/11 backbone/model-00002-of-00003.safetensors: expected 3,979,828,128 bytes; checking SHA-256...
[OpenJev] 4/11 backbone/model-00003-of-00003.safetensors: expected 440,425,856 bytes; checking SHA-256...
[OpenJev] 5/11 backbone/model.safetensors.index.json: expected 33,047 bytes; checking SHA-256...
[OpenJev] 6/11 calibration.json: expected 44 bytes; checking SHA-256...
[OpenJev] 7/11 decision_head.safetensors: expected 37,809,208 bytes; checking SHA-256...
[OpenJev] 8/11 openjev_config.json: expected 458 bytes; checking SHA-25

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[OpenJev] Load backbone/head/tokenizer; target=cuda, dtype=bfloat16: exited after 4.6s
[OpenJev] Model loaded on cuda; configuring decision branches
[OpenJev] Delta kernel=fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule; convolution=torch
[efficiency] Model ready on cuda:0
[efficiency] START K=16: 2 questions; each may take up to 4 READ decisions
openjev_frozen 1/2 status=ok ETA=1.7s peak_reserved_gib=8.0859375
openjev_frozen 2/2 status=ok ETA=0.0s peak_reserved_gib=8.0859375
[efficiency] START K=32: 2 questions; each may take up to 4 READ decisions
openjev_frozen 1/2 status=ok ETA=3.1s peak_reserved_gib=8.0859375
openjev_frozen 2/2 status=ok ETA=0.0s peak_reserved_gib=8.0859375
[efficiency] START K=64: 2 questions; each may take up to 4 READ decisions
openjev_frozen 1/2 status=ok ETA=5.8s peak_reserved_gib=8.111328125
openjev_frozen 2/2 status=ok ETA=0.0s peak_reserved_gib=8.111328125
[efficiency] START K=128: 2 questions; each may take up to 4 READ decisions
openjev_frozen 1/2 

[{'question_id': 'hotpotqa:5a710a1e5542994082a3e4fd',
  'group_id': 'hotpotqa:5a710a1e5542994082a3e4fd',
  'dataset': 'hotpotqa',
  'method': 'openjev_frozen',
  'seed': 17,
  'pool_hash': '7f11ef847bb5bd29538b28cf04684e30f79649368f3018848d2db3ec50bd2757',
  'k': 16,
  'budget': 4,
  'formal': True,
  'answer_model': None,
  'threshold': 0.5,
  'status': 'ok',
  'support_recall': 0.6666666666666666,
  'pool_support_recall': 1.0,
  'complete_support': 0.0,
  'false_stop': 1.0,
  'refused': 0.0,
  'reads': 1,
  'ndcg': None,
  'rank_evaluable_states': 0,
  'suf_evaluable_states': 2,
  'selected_ids': ['chunk-c1d05f718c4fb0a86f2f58a34a7ffa9d'],
  'suf_accuracy': 0.5,
  'suf_brier': 0.1392287197045807,
  'suf_ece': 0.2665764687117189,
  'suf_auroc': None,
  'trace': [{'selected_ids': [],
    'budget': 4,
    'h': 0.0054908390156924725,
    'action': 'chunk-c1d05f718c4fb0a86f2f58a34a7ffa9d',
    'candidate_ids': ['chunk-c1d05f718c4fb0a86f2f58a34a7ffa9d',
     'chunk-e198f28fe8a3f0ea352f5b96

In [ ]:
# Read-only audit. No model loading, API calls, or result-file modifications.
import json
from collections import Counter
expected = DIAGNOSTIC_LIMIT
report_dir = ROOT / 'results' / 'efficiency' / 'openjev_frozen'
for k in efficiency_config['evaluation']['k_values']:
    path = report_dir / f'K{k}' / 'raw.jsonl'
    if not path.exists():
        print(f'K={k}: MISSING {path}')
        continue
    with path.open(encoding='utf-8') as stream:
        counts = Counter(json.loads(line).get('status', 'unknown') for line in stream if line.strip())
    total = sum(counts.values())
    print(f'K={k}: rows={total}, expected={expected}, status={dict(counts)}')
    if total != expected:
        print('WARNING: incomplete or overwritten results; do not report this as a completed full experiment.')
meta = report_dir / 'report_metadata.json'
if meta.exists():
    saved = json.loads(meta.read_text(encoding='utf-8'))
    print('Summary rows:', saved.get('rows'), 'expected:', expected * len(efficiency_config['evaluation']['k_values']))
